# 📊 Twitch Viewer Analytics Dashboard
## Stage 7 — Streamlit Dashboard Assembly

> **Goal:** Assemble all 12 Stage 6 charts into a production-ready multi-tab
> Streamlit dashboard. Deliverables: `app.py`, `charts.py`, `.streamlit/config.toml`,
> `requirements.txt`, and `README.md`.

### What we built

| File | Purpose |
|---|---|
| `charts.py` | All 12 `build_*` functions + design system constants |
| `app.py` | Streamlit app — 6 tabs, KPI cards, sidebar, interactive widgets |
| `.streamlit/config.toml` | Dark theme matching the notebook design system |
| `requirements.txt` | Pinned dependencies |
| `README.md` | Local run + Streamlit Cloud deployment guide |

### Tab layout

| Tab | Charts | Interactive widget |
|---|---|---|
| 📈 Overview | V1 + V2 + V11 | — |
| 🎯 Loyalty | V3 + V4 + top-channel table | — |
| 🎮 Content | V5 + V6 + V12 | — |
| 📱 Platform | V7 + V8 + platform stats table | Graceful fallback if V8 data missing |
| 🔍 Discovery | V9 + discovery table | Graceful fallback if V9 data missing |
| 🔥 Binge | V10 + summary metrics + binge day table | Year selectbox |


---
## 0. Prerequisites — verify files are in place

In [ ]:
import os

STAGE7_DIR = '.'   # adjust to your stage7 folder path if needed

required_files = [
    'app.py',
    'charts.py',
    'requirements.txt',
    '.streamlit/config.toml',
    'README.md',
]

print("Stage 7 file check:")
all_ok = True
for f in required_files:
    path = os.path.join(STAGE7_DIR, f)
    ok   = os.path.exists(path)
    if not ok: all_ok = False
    size = os.path.getsize(path) if ok else 0
    print(f"  {'✅' if ok else '❌'}  {f:35s}  {size:,} bytes")

print()
print(f"{'✅ All files present' if all_ok else '❌ Some files missing'}")


---
## 1. Install dependencies

In [ ]:
# Run this once to install all required packages
# (already done if you ran pip install -r requirements.txt)
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Dependencies installed successfully")
else:
    print("❌ Installation error:")
    print(result.stderr[:500])


---
## 2. Verify charts.py imports correctly

In [ ]:
import sys
sys.path.insert(0, '.')   # ensure local charts.py is found

try:
    from charts import (
        build_v1_lifecycle, build_v2_engagement_timeline,
        build_v3_loyalty_heatmap, build_v4_channel_sunburst,
        build_v5_esports_calendar, build_v6_genre_treemap,
        build_v7_platform_divergence, build_v8_hourday_heatmap,
        build_v9_discovery_funnel, build_v10_binge_calendar,
        build_v11_yoy_waterfall, build_v12_monthly_heatmap,
        BG, TEAL, GOLD, PURPLE, RED,
    )
    functions = [f'build_v{i}' for i in range(1, 13)]
    print("✅ charts.py imported successfully")
    print(f"   {len(functions)} chart functions loaded")
    print(f"   Design system: BG={BG}, TEAL={TEAL}, GOLD={GOLD}")
except ImportError as e:
    print(f"❌ Import failed: {e}")


---
## 3. Quick smoke-test — build one chart

In [ ]:
import pandas as pd, os

DATA_DIR = os.path.join('.', 'data')
PATH_ENG = os.path.join(DATA_DIR, 'df_engineered.csv')

if os.path.exists(PATH_ENG):
    df = pd.read_csv(PATH_ENG, low_memory=False)
    df['day'] = pd.to_datetime(df['day'], errors='coerce')
    for c in ['is_lol','is_esports','is_top10_channel','is_binge_day']:
        if c in df.columns: df[c] = df[c].astype(bool)
    if 'hours_watched' not in df.columns:
        df['hours_watched'] = df['minutes_watched_unadjusted'] / 60
    df['year'] = df['year'].astype(int)

    fig = build_v1_lifecycle(df)
    fig.show()
    print("✅ V1 chart built and displayed")
else:
    print(f"⚠️  Data file not found at {PATH_ENG}")
    print("   Place your CSV files in the data/ folder before running app.py")


---
## 4. Run the Streamlit app

In [ ]:
# This cell shows you the command to run — execute it in your terminal, not here
print("Run this command in your terminal:")
print()
print("  cd path/to/your/stage7/folder")
print("  streamlit run app.py")
print()
print("The app will open automatically at http://localhost:8501")
print()
print("To stop the app: press Ctrl+C in the terminal")


---
## 5. Streamlit Cloud deployment

In [ ]:
# Summary of deployment steps — see README.md for full details
steps = [
    "1. Push project to GitHub (exclude data/ if data is sensitive)",
    "2. Go to share.streamlit.io and sign in with GitHub",
    "3. Click 'New app' → select repo → set file = app.py → Deploy",
    "4. Your app gets a public URL: yourname-twitch-analytics-app-XXXX.streamlit.app",
    "5. Share that URL with anyone — works on any device, no setup needed",
]
print("Streamlit Cloud deployment steps:")
for s in steps:
    print(f"  {s}")
print()
print("Total time from git push to live URL: ~3 minutes")


---
## Stage 7 Summary

### ✅ Delivered

| File | Lines | Purpose |
|---|---|---|
| `app.py` | ~270 | Full Streamlit dashboard — 6 tabs, KPI cards, sidebar, caching |
| `charts.py` | ~430 | All 12 `build_*` functions + design system |
| `.streamlit/config.toml` | 8 | Dark theme (#0f0f1a background) |
| `requirements.txt` | 4 | streamlit, pandas, numpy, plotly |
| `README.md` | ~110 | Local run + Streamlit Cloud guide |

### 📐 Key design decisions

- **`@st.cache_data` on every chart** — charts are computed once and cached; switching tabs is instant
- **`_` prefix on DataFrame arguments** — tells Streamlit to skip hashing large DataFrames for performance
- **Graceful V8/V9 fallback** — if `df_video_play_clean.csv` is missing, those tabs show a clear info message instead of crashing
- **Dark CSS overrides** — `backgroundColor`, `secondaryBackgroundColor`, and tab styles all match the `#0f0f1a` design system from Stage 6
- **Binge year selectbox** — the only interactive filter; changing year invalidates only the V10 cache, not all 12 charts

### 🚀 What's next (Stage 8 ideas)

- Add a **date range slider** in the sidebar to filter the entire dataset
- Add a **channel search** input to V4 sunburst
- Add **export to PDF** button using `kaleido`
- Add a **"Stats for Nerds"** tab with raw data tables and CSV download
